# Daily Merge

Merge raw session CSV files into one `merged_<animal>.csv` file per animal, then optionally merge all animals into `merged_all_subjects.csv` for the full cohort of the selected line.

## 1. Setup

Run this cell first. It makes imports work whether the notebook is launched from the repo root or from inside `notebooks/ASD` or `notebooks/Stakes`.

In [29]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "analysis" / "daily_merge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/mafaldavalente/Documents/Mafalda_analysis')

## 2. Choose Dataset

Set `RAT = None` to process every animal in the cohort, or set it to one animal ID such as `"JCS0013"`.

In [30]:
LINE = "Stakes"
COHORT = "cohort2"
RAT = None  # e.g. "JCS0013", or None for all animals

MODE = "both"  # "session", "animals", or "both"
MODEL_FILE = None

## 3. Optional Session Removals

Edit `SESSION_EDITS` before merging if a daily CSV should be removed entirely, or if only the bad tail/range of a session should be removed or marked repeated. File names must match the raw CSV names inside each animal folder.


In [31]:
# Optional per-animal cleanup rules applied while creating merged_<animal>.csv.
# These affect the merged output only; they do not edit the raw daily CSV files.
#
# Actions:
# - drop_entire_session: skip that raw CSV completely
# - drop_from_trial: remove trials with trial >= start_trial
# - drop_trial_range: remove trials from start_trial through end_trial, inclusive
# - drop_block: remove one or more block values from a session file
# - mark_repeated_from: keep rows but set repeated_trial = True from start_trial onward

SESSION_EDITS = {
    # Examples for ASD0019. Uncomment/edit the raw filenames and thresholds as needed.
    # "ASD0019": [
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_entire_session"},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_from_trial", "start_trial": 5000},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_trial_range", "start_trial": 1000, "end_trial": 1500},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "block": 2},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "blocks": [2, 3]},
    # ],
    "JCS0020": [
         {"file": "out_JCS0020_260716.csv", "action": "drop_entire_session"},
        ],
}

SESSION_EDITS


{'JCS0020': [{'file': 'out_JCS0020_260716.csv',
   'action': 'drop_entire_session'}]}

## 4. Optional Bad RT Values

Use `RT_VALUE_EDITS` when task outcomes and abort labels are valid, but the recorded numeric `timed_rt` values should be ignored in RT analyses. These edits keep the trials and only set `timed_rt` to missing in the merged outputs.


In [32]:
# Numeric RT values to ignore while keeping trials for accuracy/choice/abort analyses.
# This only blanks timed_rt and adds rt_value_valid / rt_value_note columns.
# It does not change success, abort_type, choices, trial counts, or repeated_trial.

RT_VALUE_EDITS = [
    {
        "setup": 2,
        "start_date": "2026-06-13",
        "end_date": "2026-06-18",  # update if the setup-2 issue continues
        "date_col": "source_date",
        "setup_col": "box",
        "rt_col": "timed_rt",
        "reason": "setup 2 RT value recording issue",
    },

]

RT_VALUE_EDITS


[{'setup': 2,
  'start_date': '2026-06-13',
  'end_date': '2026-06-18',
  'date_col': 'source_date',
  'setup_col': 'box',
  'rt_col': 'timed_rt',
  'reason': 'setup 2 RT value recording issue'}]

## 5. Preview Animals

Check which animals will be processed before writing merged files.

In [36]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import get_animals_for_cohort, get_base_dir

base_dir = get_base_dir(LINE, COHORT)
animals = get_animals_for_cohort(LINE, COHORT, rat=RAT)

print(f"Base directory: {base_dir}")
print(f"Animals ({len(animals)}): {animals}")

Base directory: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/Stakes_cohort2
Animals (10): ['JCS0013', 'JCS0014', 'JCS0015', 'JCS0016', 'JCS0017', 'JCS0018', 'JCS0019', 'JCS0020', 'JCS0021', 'JCS0022']


## 6. Merge Daily Files Per Animal

This creates or updates `merged_<animal>.csv` files in the cohort data folder.

In [37]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import merge_session_files

if MODE in ("session", "both"):
    merge_session_files(
        line=LINE,
        cohort=COHORT,
        rat=RAT,
        session_edits=SESSION_EDITS,
        rt_value_edits=RT_VALUE_EDITS,
    )
else:
    print("Skipping per-animal session merge.")

Processing 10 animal(s) for Stakes cohort2: JCS0013, JCS0014, JCS0015, JCS0016, JCS0017, JCS0018, JCS0019, JCS0020, JCS0021, JCS0022
Using latest file 'out_JCS0013_260720.csv' as column reference (79 columns).
Total unique columns across all files: 79
✅ Added out_JCS0013_260706.csv (129 rows)
✅ Added out_JCS0013_260707.csv (342 rows)
✅ Added out_JCS0013_260709.csv (534 rows)
✅ Added out_JCS0013_260711.csv (604 rows)
✅ Added out_JCS0013_260713.csv (554 rows)
✅ Added out_JCS0013_260715.csv (543 rows)
✅ Added out_JCS0013_260716.csv (732 rows)
✅ Added out_JCS0013_260717.csv (708 rows)
✅ Added out_JCS0013_260718.csv (658 rows)
✅ Added out_JCS0013_260720.csv (563 rows)
🧹 Set timed_rt=NaN for 0 rows from 2026-06-13 to 2026-06-18 on setup 2
🎉 Merged 10 files into /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/Stakes_cohort2/merged_JCS0013.csv
Final shape: (5367, 85)

Session order check:
   __new_session           __source_file __session_sort_key
0              1  out_JCS0013_26070

## 7. Merge Animals Into Cohort File

This creates or updates `merged_all_subjects.csv` in the cohort data folder.

In [38]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import merge_subject_files_with_model

if MODE in ("animals", "both"):
    merged_df = merge_subject_files_with_model(
        line=LINE,
        cohort=COHORT,
        model_file=MODEL_FILE,
    )
else:
    merged_df = None
    print("Skipping cohort-level animal merge.")

if merged_df is not None:
    display(merged_df.head())
    print(merged_df.shape)

📘 Using model file 'merged_JCS0013.csv' with 85 columns.
🧾 Total unique columns across all subjects: 92
✅ Added merged_JCS0013.csv (5367 rows)
✅ Added merged_JCS0014.csv (4513 rows)
✅ Added merged_JCS0015.csv (2762 rows)
✅ Added merged_JCS0016.csv (3777 rows)
✅ Added merged_JCS0017.csv (3158 rows)
✅ Added merged_JCS0018.csv (2103 rows)
✅ Added merged_JCS0019.csv (3601 rows)
✅ Added merged_JCS0020.csv (3809 rows)
✅ Added merged_JCS0021.csv (2370 rows)
✅ Added merged_JCS0022.csv (3684 rows)
🎉 Saved merged dataset: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/Stakes_cohort2/merged_all_subjects.csv
Final shape: (35144, 92)


,animal,batch,experimenter,version,bias,repeated_trial,trial,trial_start,tared_trial_start,trial_end,...,source_date,rt_value_valid,rt_value_note,trial_start_frame,trial_end_frame,cnp_start_frame,rt_start_frame,mt_start_frame,lnp_start_frame,lnp_end_frame
0,JCS0013,stakes,MV,0.10.0,0.00,True,1,3.866179e+09,0.000000,3.866179e+09,...,2026-07-06,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JCS0013,stakes,MV,0.10.0,0.04,True,2,3.866179e+09,182.144000,3.866179e+09,...,2026-07-06,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JCS0013,stakes,MV,0.10.0,0.04,True,3,3.866179e+09,210.571008,3.866179e+09,...,2026-07-06,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JCS0013,stakes,MV,0.10.0,0.00,False,4,3.866179e+09,392.605984,3.866179e+09,...,2026-07-06,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JCS0013,stakes,MV,0.10.0,-0.04,False,5,3.866179e+09,434.404992,3.866179e+09,...,2026-07-06,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(35144, 92)
